In [36]:
import pandas as pd

In [37]:
df = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")
df = df.dropna(how="all").reset_index(drop=True)

In [38]:
df.head()

,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2025-09-01,Monday,Long,57%,Balanced,Balance,Invalid,NaN,Positive ORG is modest and the overnight range...
1,2025-09-02,Tuesday,Short,72%,Strong Down,Trend Continuation,Long,False,"Very large negative ORG, an overnight range we..."
2,2025-09-03,Wednesday,Long,65%,Strong Up,Exhaustion,Long,True,Large positive ORG and a sustained recovery fr...
3,2025-09-04,Thursday,Short,59%,Moderate Down,Balance,Long,False,ORG is modestly positive and the higher-timefr...
4,2025-09-05,Friday,Long,68%,Strong Up,Trend Continuation,Short,False,Large positive ORG and a broad overnight advan...


In [39]:
df["Confidence Numeric"] = (
    df["Confidence"]
    .str.rstrip("%")
    .astype(int)
)

In [40]:
confidence_accuracy = (
    df.dropna(subset=["Correct"])
      .groupby("Confidence Numeric")["Correct"]
      .agg(["count", "mean"])
)

confidence_accuracy["Accuracy %"] = confidence_accuracy["mean"] * 100

confidence_accuracy

,count,mean,Accuracy %
Confidence Numeric,,,
53,1,0.0,0.0
54,2,0.0,0.0
55,2,1.0,100.0
56,4,0.75,75.0
57,16,0.5625,56.25
58,29,0.655172,65.517241
59,20,0.6,60.0
60,13,0.538462,53.846154
61,18,0.833333,83.333333


In [41]:
valid_df = df.dropna(subset=["Correct"]).copy()

valid_df["Confidence Group"] = pd.cut(
    valid_df["Confidence Numeric"],
    bins=[0, 58, 62, 65, 100],
    labels=["≤58", "59–62", "63–65", "≥66"]
)

In [42]:
confidence_groups = (
    valid_df.groupby("Confidence Group", observed=True)["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_groups["Accuracy %"] = confidence_groups["mean"] * 100

confidence_groups

,count,sum,mean,Accuracy %
Confidence Group,,,,
≤58,54,33,0.611111,61.111111
59–62,70,44,0.628571,62.857143
63–65,36,20,0.555556,55.555556
≥66,73,41,0.561644,56.164384


In [43]:
valid_df["Confidence Half"] = valid_df["Confidence Numeric"].apply(
    lambda x: "≤62" if x <= 62 else ">62"
)

confidence_halves = (
    valid_df.groupby("Confidence Half")["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_halves["Accuracy %"] = confidence_halves["mean"] * 100

confidence_halves

,count,sum,mean,Accuracy %
Confidence Half,,,,
>62,109,61,0.559633,55.963303
≤62,124,77,0.620968,62.096774


In [44]:
overall_accuracy = valid_df["Correct"].mean() * 100

print(f"Overall accuracy: {overall_accuracy:.2f}%")
print(f"Correct: {valid_df['Correct'].sum()} / {len(valid_df)}")

Overall accuracy: 59.23%
Correct: 138 / 233
